# Revisiting SASRec on Amazon Reviews 2023

**Team:** Alina Chuang, Frank Bartkus, Kyle Nguyen, Matthew Mikhin

We re-run the experiments of SASRec (Kang & McAuley, ICDM 2018) on the 2023 release of the
Amazon review data and check whether the paper's conclusions still hold.

This notebook is the whole project and runs from top to bottom.

- **The rules everyone follows** are in the TODO box of the member who builds that part, so
  that every method is trained and measured in the same way. Section 2 lists where each is.
- **Every other section starts with a TODO list for its owner.** Delete a TODO item when it is
  done; delete the whole TODO box when the section is finished.

**Contents**

0. About: how to run, who does what, contribution statement, use of AI tools
1. Problem definition
2. Settings
3. Data
4. Evaluation
5. Methods
6. Training and tuning
7. Results
8. Ablations: which parts of SASRec matter?
9. Sampled evaluation vs full ranking
10. Demo
11. Discussion: observations, limitations, future work
12. References

## 0. About this notebook

### How to run it

```bash
uv sync                  # once: installs the Python packages from uv.lock into .venv
uv run jupyter lab       # or open the notebook in VS Code and pick the .venv kernel
```

Then run all cells from the top. With `RUN_EXPERIMENTS = False` (Section 2) the trained
results are read from `results/` and the notebook runs in minutes; with `True` every model is
trained again, which takes hours.

How we work on this one file together (git, branches, outputs) is in `docs/team_spec.md`.

### Who does what, and in which order

Hand-in: **Monday, November 30, 23:55**. Presentations: December 1 and 3.

| # | Task | Who | Needs first | Target |
|---|---|---|---|---|
| 0 | The rules every method follows, written into each owner's section (list in Section 2) | Alina | — | **done** Wed Nov 18; everyone reads them by Thu Nov 19 |
| 1 | Propose the category (Section 3.2) | Matthew (team agrees) | — | Thu Nov 19 |
| 2 | Data files in `data/`, `load_data`, `leave_one_out`, data checks (Section 3.1) | Frank | 1 | Sat Nov 21 |
| 3 | Negatives, metrics, `recommend`, `check_model`, sanity checks (Section 4) | Frank | 2 | Sun Nov 22 |
| 4 | PopRec (Section 5.1) | Frank | 3 | Sun Nov 22 |
| 5 | Problem definition, related work, the paper's numbers (Section 1) | Kyle | — | Sun Nov 22 |
| 6 | SASRec with its three ablation switches, and the training loop, on fake data (Sections 5.3, 6) | Alina | — | Sun Nov 22 |
| 7 | Data checks and EDA (Section 3.2) | Matthew | 2 | Mon Nov 23 |
| 8 | BPR, FMC, FPMC, tuned on validation, test with 3 seeds (Section 5.2) | Kyle | 3, training loop | Wed Nov 25 |
| 9 | SASRec tuned on validation, test with 3 seeds, model saved for the demo | Alina | 3 | Wed Nov 25 |
| 10 | Ablations (Section 8) | Alina | 9 | Fri Nov 27 |
| 11 | Results table, sampled vs full (Sections 7, 9) | Frank | 8, 9 | Sat Nov 28 |
| 12 | Answer RQ1 (Section 7) | Kyle | 11 | Sat Nov 28 |
| 13 | Figures (Sections 3.2, 7) | Matthew | 11 | Sat Nov 28 |
| 14 | Demo cell and demo video (Section 10) | Matthew | 4, 9 | Sun Nov 29 |
| 15 | Discussion, contribution statement, AI note (Sections 0, 11) | everyone | 11 | Sun Nov 29 |
| 16 | Final run with outputs; hand in the notebook and the video | Matthew | everything | Mon Nov 30 |

Steps 1 to 3 block everyone else. Please do them first.

### Contribution statement

> **TODO (everyone, at the end).** One row per member, agreed by all four.

| Member | Contribution |
|---|---|
| Alina Chuang | |
| Frank Bartkus | |
| Kyle Nguyen | |
| Matthew Mikhin | |

### Use of generative AI tools

> **TODO (everyone).** The course requires a short note on which AI tools each of us used and
> for what. Each member writes their own row.

| Member | Tool | Used for |
|---|---|---|
| Alina Chuang | | |
| Frank Bartkus | | |
| Kyle Nguyen | | |
| Matthew Mikhin | | |

## 1. Problem definition

> **TODO (Kyle)**
> - [ ] What a sequential recommender does, with a small example of one user's history and
>   the next item.
> - [ ] SASRec in one paragraph. Alina sends you her method paragraph (Section 5.3).
> - [ ] Why re-test on the 2023 data: what changed since the paper's data.
> - [ ] The three research questions RQ1 to RQ3 (`README.md`).
> - [ ] The paper's numbers that we compare with, as a table here: Hit@10 and NDCG@10 of
>   PopRec, BPR, FMC, FPMC and SASRec on Beauty and Games (Table III), and of the three
>   ablations we run (Table IV). Frank's table (Section 7), your RQ1 answer and Alina's
>   ablations (Section 8) put our numbers next to these.
> - [ ] Related work: SASRec ([1808.09781](https://arxiv.org/abs/1808.09781)), Amazon Reviews
>   2023 ([2403.03952](https://arxiv.org/abs/2403.03952)), why baselines must be tuned
>   ([1907.06902](https://arxiv.org/abs/1907.06902)), later loss improvements for SASRec
>   ([2309.07602](https://arxiv.org/abs/2309.07602)).

## 2. Settings (Alina)

SASRec is compared with PopRec, BPR, FMC and FPMC. That comparison only means something if
every method gets the same data, the same test cases, the same negatives, the same metric and
the same tuning. A difference in any of these would show up as a difference between methods.
So the team fixes these rules once, and every section follows them.

Alina drafted the rules. They are final once every member has read them and nobody objects
(by Thursday, November 19). Each rule is written in the TODO box of the member who builds that
part, because that member explains it in the report and in the presentation:

| Rules for | Section | Owner |
|---|---|---|
| reading the data and splitting it | 3.1 | Frank |
| negatives, ranks, metrics, the checks every model must pass | 4 | Frank |
| what every model looks like | 5 | Matthew |
| training, tuning, early stopping | 6 | Alina |
| result files and method names | 7 | Frank |
| names of the ablations | 8 | Alina |

If you think a rule should change, tell the team first; do not change it on your own.

This section holds only the fixed numbers that several sections use, and `set_seed`. They come
first because the code of every later section uses them.

In [ ]:
import random

import torch

# Which Amazon category we use: Matthew proposes, the team agrees (Section 3.2).
CATEGORY = None

# True: train every model again (hours). False: read the saved results from results/.
RUN_EXPERIMENTS = False

# Evaluation, as in the paper (Section IV-D)
EVAL_SEED = 0  # one fixed set of 100 negatives, the same for every method
N_NEGATIVES = 100  # each answer is ranked against 100 items the user never touched
K = 10  # we report Hit@10 and NDCG@10

# Training
TRAIN_SEEDS = [1, 2, 3]  # every trained model runs 3 times; we report mean and std
PATIENCE = 20  # stop after 20 epochs without a better validation NDCG@10 (IV-C)
MAX_EPOCHS = 200  # never train longer than this, so every run ends
MAX_LEN = 50  # SASRec reads at most the last 50 items (paper IV-C)

Add the imports you need to the cell above, so all imports are in one place.

A model uses random numbers for its starting weights, for picking negative items and for the
order of the batches. If two runs with the same seed gave different results, our mean over 3
seeds would mean nothing. `set_seed` puts every random number generator we use back to the
same starting point. The training loop calls it before it creates a model.

In [ ]:
def set_seed(seed):
    """Fix every random number generator we use, so a run can be repeated exactly."""
    random.seed(seed)  # Python's own generator, for example random.sample
    torch.manual_seed(seed)  # torch on the CPU, and on the GPU too (CUDA or Apple MPS)


# Check: the same seed gives the same numbers.
set_seed(1)
first = torch.rand(3)
set_seed(1)
second = torch.rand(3)
print(first)
print(second)
print("same:", torch.equal(first, second))

## 3. Data

### 3.1 Source and loading (Frank)

> **TODO (Frank)** The team agreed on the rules in the tables below, so that every method gets
> exactly the same data. Build the loading by following them exactly. Then replace the tables
> with your own explanation: each rule and why, in your own words, because you present this
> part. If you think a rule should change, tell the team first.
>
> | Decision | Rule | Why |
> |---|---|---|
> | Source | the official 5-core `last_out_w_his` files of `CATEGORY` (train, valid, test); no other filtering | the paper's 5-core data and leave-one-out split (Section IV-A) |
> | Interaction | every row is one interaction; the star rating is ignored | the paper treats every review as implicit feedback (IV-A) |
> | Repeated user–item pairs | kept as they are in the files; Matthew reports how many there are (Section 3.2) | the official files are the protocol; changing them would make our data differ from everyone else's |
> | Order of one user's items | the user's train rows sorted by `timestamp`, **stable sort** (rows with the same timestamp keep their file order); then the valid item; then the test item | rows with the same timestamp must come out in the same order on every computer |
> | User indices | 0, 1, 2, … in order of first appearance in the train file | the same indices on every computer, with no mapping file |
> | Item indices | 0, 1, 2, … in order of first appearance, reading the train file, then valid, then test | same reason |
> | Validation | answer = second-to-last item; input history = all items before it | leave-one-out (IV-A) |
> | Test | answer = last item; input history = all items before it, **including the validation item** | the paper uses the validation item as input at test time (IV-A) |
> | The official `history` column | used only to check our sequences, never as model input | we build every history from our own sequences, so all sections use exactly the same ones |
>
> The names every later section uses:
>
> | Name | What it is |
> |---|---|
> | `sequences` | `sequences[u]`: item indices of user `u`, oldest first: train items, valid item, test item |
> | `user_ids`, `item_ids` | the original ids: `user_ids[u]` is the `user_id` of user `u`, `item_ids[j]` the `parent_asin` of item `j` |
> | `n_users`, `n_items` | `len(user_ids)`, `len(item_ids)` |
> | `train_seqs[u]` | `sequences[u][:-2]`: what models learn from, and the validation input history |
> | `valid_answers[u]` | `sequences[u][-2]` |
> | `test_histories[u]` | `sequences[u][:-1]` |
> | `test_answers[u]` | `sequences[u][-1]` |
>
> Item index `j` is column `j` of every score matrix.
>
> - [ ] Download the three files of `CATEGORY` into `data/<CATEGORY>/` and commit them
>   (GitHub refuses files over 100 MB). Describe the source here.
> - [ ] `load_data(category)` returns `sequences, user_ids, item_ids`, following the rules above.
> - [ ] `leave_one_out(sequences)` returns `train_seqs, valid_answers, test_histories,
>   test_answers`, and raises an error if a user has fewer than 3 items.
> - [ ] Checks that stop the notebook with a clear message (`raise`, not `print`):
>   every user has exactly one valid row and one test row; every user has at least one train
>   row; the valid row's `history` equals the user's train items in our order.

### 3.2 Data checks and exploratory analysis (Matthew)

> **TODO (Matthew)**
> - [ ] **First, before everyone else can start:** the 5-core sizes (users, items,
>   interactions, file sizes) of the 2023 categories closest to the paper's Beauty and Games.
>   Propose one category. Every file must be under 100 MB for GitHub. Not
>   `All_Beauty`: its 5-core version has only 253 users.
> - [ ] The five columns of the paper's Table II for our data, next to the paper's numbers.
> - [ ] Interactions per user and per item, log scale on both axes; mark 50 on the user plot
>   and give the share of users above 50 to Alina (SASRec reads at most `MAX_LEN = 50`).
> - [ ] Repeated user–item pairs, time range, unit of `timestamp`, whether the train, valid and
>   test files have the same columns, and whether any user's timestamps go backwards.

## 4. Evaluation (Frank)

> **TODO (Frank)** The team agreed on the rules in the tables below, so that every method is
> measured in exactly the same way. Build the evaluation by following them exactly. Then
> replace the tables with your own explanation: each rule and why, in your own words, because
> you present this part. If you think a rule should change, tell the team first.
>
> | Decision | Rule | Why |
> |---|---|---|
> | Negatives | for every user, separately for validation and for test: 100 items drawn uniformly, without repeats, from the items that appear **nowhere** in the user's sequence | the paper's sampled evaluation (IV-D); an item the user did interact with is not a fair negative |
> | Random generator for negatives | `random.Random(EVAL_SEED)`, users in index order, all validation negatives first, then all test negatives | the same negatives on every computer and for every method |
> | Candidate format | a tensor of shape `(n_users, 101)`: row `u` is user `u`, **column 0 is the answer**, columns 1 to 100 the negatives | the metric code finds the answer without a search |
> | Sampled rank `r` | 1 + the number of negatives whose score is **≥** the answer's score | ties count against the answer, so a model that gives every item the same score gets rank 101, not 1 |
> | Full-ranking rank `r` | 1 + the number of other items, not in the input history, whose score is **≥** the answer's score | the same tie rule over all items; items the user already has are left out, except the answer itself |
> | Hit@10 | 1 if `r ≤ 10`, else 0; averaged over users | paper IV-D |
> | NDCG@10 | `1 / log2(r + 1)` if `r ≤ 10`, else 0; averaged over users | paper IV-D; with one answer this is the whole formula |
>
> The two checks every model must pass, built as `check_model`:
>
> | Check | Why |
> |---|---|
> | `score` gives shape `(n_users, n_items)`, all finite | the evaluation reads exactly this; `nan` breaks every metric |
> | the same seed gives the same scores | otherwise the mean over 3 seeds means nothing |
>
> - [ ] The negatives: `valid_candidates` and `test_candidates`, each of shape
>   `(n_users, 101)` with the answer in column 0.
> - [ ] `evaluate_sampled(model, histories, candidates)` returns `{"hit@10": ..., "ndcg@10": ...}`.
>   The training loop calls it after every epoch.
> - [ ] `evaluate_full(model, histories, answers)`: the same metrics with the full-ranking rank.
> - [ ] `recommend(model, users, histories, k)`: the top `k` items per case with the input
>   history removed, for the demo (Section 10). Hint: write `-inf` into a copy of the
>   scores, never into the tensor the model returned (PopRec can return its own stored scores).
> - [ ] `check_model(ModelClass, **settings)`: runs the two checks above on small fake data
>   and prints one line if all pass. Every model owner runs it in the cell right after their
>   class.
> - [ ] Sanity checks of the metric code, kept as a cell: a model that gives every item the
>   same score gets 0; a model that gives the answer the highest score gets 1.

## 5. Methods

> **TODO (Matthew)** The team agreed on the rules below, so that one training loop and one
> evaluation work for every method. Every model owner follows them. Replace them with your own
> explanation, because you present how the parts of the notebook fit together.
>
> Every method (PopRec, BPR, FMC, FPMC, SASRec) is a class with three methods:
>
> | Method | What it does |
> |---|---|
> | `__init__(self, n_users, n_items, ...)` | creates the model; settings such as `d` come after the first two arguments, with the paper's value as the default |
> | `train_one_epoch(self, train_seqs)` | learns from the training sequences once and returns the average loss as a `float` |
> | `score(self, users, histories)` | returns a tensor of shape `(len(users), n_items)`: a score for every item for every case; higher means recommend earlier |
>
> Case `r` is user `users[r]` with input history `histories[r]`. BPR and FPMC need the user;
> FMC and SASRec need the history; PopRec needs neither.
>
> Rules:
>
> 1. A model learns only from what `train_one_epoch` receives. Never read a global variable
>    such as `test_answers` or `sequences` inside a model.
> 2. Never call `set_seed` inside a model, and never create a random generator with a fixed
>    number inside a model. The training loop seeds everything before the model is created.
> 3. Every setting is an argument of `__init__`; no numbers hidden inside methods. Ablations
>    are settings too, for example `use_pe=False`.
> 4. `score` returns exactly `n_items` columns. A padding item (SASRec) is removed before
>    returning.
> 5. No metric code inside a model; Section 4 measures every model.
> 6. CPU by default. A `device` argument is allowed; `score` returns a CPU tensor.
>
> - [ ] Before the final run: check every class against these rules, and that every class
>   passes `check_model` (Section 4).

For each method, its owner writes a short explanation above the code: the idea, why it is in
the comparison, and its merits and limits.

### 5.1 PopRec (Frank)

> **TODO (Frank)**
> - [ ] PopRec: every item's score is how often it appears in `train_seqs`; the same scores
>   for every user.
> - [ ] `check_model`.

### 5.2 BPR, FMC and FPMC (Kyle)

> **TODO (Kyle)**
> - [ ] BPR: `x(u, i) = P[u] · Q[i]`. FMC: `x(l, i) = M[l] · N[i]`, where `l` is the previous
>   item. FPMC: the sum of both.
> - [ ] Train all three with the BPR loss `−ln σ(x(u, i) − x(u, j)) + l2 · (squared norms)`,
>   one negative `j` per positive `i`, drawn from items not in the user's train sequence.
>   Hint: `-torch.nn.functional.logsigmoid(x)` stays finite where `-log(sigmoid(x))` becomes
>   `inf`.
> - [ ] `score`: BPR uses `users`; FMC uses the last item of each history (for test that is
>   the validation item); FPMC uses both.
> - [ ] Choose and justify the optimizer, learning rate, batch size and starting weights.
> - [ ] `check_model` for all three; grid search of Section 6 with the shared training loop.
> - [ ] Note: in the paper BPR loses to PopRec on Beauty (Hit@10 0.3775 vs 0.4003). If yours
>   does too, first confirm the grid search ran, then report it as a finding.

### 5.3 SASRec (Alina)

> **TODO (Alina)**
> - [ ] The model with the paper's settings (Sections III and IV-C): 2 self-attention blocks,
>   1 head, learned positional embeddings, item embeddings shared between input and output,
>   at most `MAX_LEN = 50` items, dropout 0.5, Adam with learning rate 0.001, batch size 128,
>   binary cross-entropy with one sampled negative per position.
> - [ ] The ablation switches of Section 8: `use_pe`, `share_item_emb`, `n_blocks`.
> - [ ] `check_model` with the default settings and with every switch.
> - [ ] The explanation: how self-attention reads a sequence, why it is causal, merits and
>   limits compared with the Markov-chain baselines (FMC, FPMC).
> - [ ] Send the method paragraph to Kyle for Section 1.

## 6. Training and tuning (Alina)

> **TODO** The team agreed on the rules below, so that every trained model is tuned in the
> same way and the baselines get as much tuning as SASRec. Alina builds `train_model` by
> following them, replaces them with her own explanation and presents this part. Kyle follows
> the same rules for his models.
>
> | Model | Tuned on validation | Grid |
> |---|---|---|
> | BPR, FMC, FPMC | `d`, `l2` | `d` ∈ {10, 30, 50}; `l2` ∈ {0.001, 0.01, 0.1}: 9 points |
> | SASRec | `d` | {10, 30, 50}; everything else at the paper's values |
>
> The paper tries `d` ∈ {10, 20, 30, 40, 50} and `l2` ∈ {0.0001, …, 1} (IV-C). We keep points
> inside those ranges but use fewer of them, so that all training fits on our laptops; this
> goes into the limitations (Section 11).
> | PopRec | nothing | — |
>
> 1. Train every grid point with seed 1 and measure it on validation only.
> 2. Pick the point with the highest sampled validation NDCG@10. If two points tie to four
>    decimals, take the smaller `d`, then the smaller `l2`.
> 3. Train the chosen point with seeds 1, 2 and 3 and measure it on test. **Test is used only
>    here, once.**
>
> Early stopping, for every trained model: after every epoch, measure sampled validation
> NDCG@10; stop after `PATIENCE` epochs without improvement, or at `MAX_EPOCHS`; keep the
> model of the best epoch. Full ranking is reported, never used to choose anything.
>
> - [ ] **Alina:** `train_model(ModelClass, settings, seed)`: calls `set_seed`, creates the
>   model, trains with the early stopping above, and returns the best model with the number of
>   epochs. Kyle uses the same function for his models.
> - [ ] **Everyone with a trained model:** the grid search and the 3 test seeds above, saved to
>   the CSV files of Section 7, with your full grid table shown (one row per grid point), so the
>   reader sees the baselines were tuned too, not only SASRec.
> - [ ] **Alina:** save the chosen SASRec (seed 1) with `torch.save(model.state_dict(), ...)`
>   as `results/sasrec_demo.pt`, for the demo (Section 10).
> - [ ] **Frank:** PopRec on test (no tuning).

## 7. Results (Frank)

> **TODO** The team agreed on the file format and names below, so that one table can read
> every member's results. Frank replaces them with his own explanation and presents this part.
>
> Every training run adds rows to small CSV files in `results/`, committed to git. With
> `RUN_EXPERIMENTS = False`, Sections 6 to 9 read these files instead of training.
>
> | File | One row per | Columns |
> |---|---|---|
> | `results/tuning.csv` | grid point | `method, d, l2, valid_ndcg10, epochs` |
> | `results/test.csv` | method, seed, protocol | `method, seed, protocol, hit10, ndcg10, epochs` |
>
> `protocol` is `sampled` or `full`. `l2` is empty for models without it. Method names:
> `poprec`, `bpr`, `fmc`, `fpmc`, `sasrec`; the names of the ablations are in Section 8.
>
> - [ ] **Frank:** the main table, like the paper's Table III: one row per method, Hit@10 and
>   NDCG@10, mean ± std over the 3 seeds, sampled and full ranking.
> - [ ] **Kyle:** answer RQ1: does SASRec still beat the non-neural baselines? Compare with the
>   paper's numbers (Section 1).
> - [ ] **Matthew:** figures for the table.

## 8. Ablations: which parts of SASRec matter? (Alina)

> **TODO (Alina)** The three ablations of the proposal, with the names the team agreed on for
> the CSV files of Section 7:
>
> | Variant | Setting | Method name |
> |---|---|---|
> | no positional embedding | `use_pe=False` | `sasrec_nope` |
> | separate input and output item embeddings | `share_item_emb=False` | `sasrec_unshared` |
> | 0 / 1 / 3 blocks (2 is the default) | `n_blocks=0`, `1`, `3` | `sasrec_blocks0`, `sasrec_blocks1`, `sasrec_blocks3` |
>
> - [ ] Every variant with the 3 test seeds.
> - [ ] Every result next to the paper's number for the same variant (Kyle's table in
>   Section 1), and answer RQ2.

## 9. Sampled evaluation vs full ranking (Frank)

> **TODO (Frank)**
> - [ ] RQ3: is the order of the methods the same under the sampled protocol and full
>   ranking? Show both orders side by side and explain any change.

## 10. Demo (Matthew)

> **TODO (Matthew)** The course requires a recorded demo video that shows the recommender
> working. It is played in class and paused to comment. Walking through the code is not a demo.
> The whole presentation, video included, must fit in 20 minutes, so aim for about 3 minutes of
> video. The video must show the version we hand in on November 30.
>
> First build the demo here:
>
> - [ ] Product titles: the 5-core files have only `parent_asin`. Download the metadata file of
>   `CATEGORY` from the Amazon Reviews 2023 page, keep only `parent_asin` and `title` of our
>   items, save that as `data/<CATEGORY>/titles.csv.gz` and commit it (not the full metadata
>   file).
> - [ ] Load SASRec from `results/sasrec_demo.pt` (Alina saves it, Section 6) and build PopRec
>   (Section 5.1).
> - [ ] `show_user(u, history)`: prints the history as titles, oldest first, then the top 10 of
>   SASRec and the top 10 of PopRec from `recommend` (Section 4), and marks the user's real next
>   item (`test_answers[u]`) if it is in a list.
> - [ ] Pick the users before recording: two users with clearly different histories where
>   SASRec finds the real next item, and one user where it does not. Say in the video that
>   they were picked.
>
> Then record it. The scenes, in this order; each one is a place to pause and comment:
>
> 1. One sentence: "Given what a user reviewed so far, we recommend the next product."
> 2. User A: the history, then SASRec's top 10 with the real next item marked.
> 3. The same user with PopRec: the same list for every user. This is the baseline.
> 4. User B, with different taste: SASRec's list changes; PopRec's does not.
> 5. User A again, with the same items in reversed order. If SASRec's list changes, the model
>    uses the order, not only which items. If it barely changes, say so: that links to the
>    ablation without positional embeddings (Section 8).
> 6. The user where SASRec misses: an honest example of its limits.
>
> - [ ] Show only outputs: run the cells and scroll to the results; do not explain code.
> - [ ] Record the screen in 1080p (macOS: QuickTime, File → New Screen Recording; Windows:
>   Win + Alt + R). A voice-over is optional, because we comment live in class. Put a short
>   title line above each scene, so the video also makes sense when the teacher watches it on
>   Moodle.
> - [ ] Save it as `demo.mp4` and upload it to Moodle with the hand-in. Do not commit it to git
>   (too large).
> - [ ] Agree with the team on who comments at which pause.

## 11. Discussion

> **TODO**
> - [ ] **Everyone:** observations, limitations and future work for your own part.
> - [ ] **Matthew:** first draft of the conclusion: the answers to RQ1 to RQ3, which
>   conclusions agree with the paper, which do not, and possible reasons. Everyone reviews it.
> - [ ] **Everyone:** the contribution statement and the AI note in Section 0.

## 12. References

- Wang-Cheng Kang and Julian McAuley. *Self-Attentive Sequential Recommendation.* ICDM 2018.
  [arXiv:1808.09781](https://arxiv.org/abs/1808.09781)
- Yupeng Hou, Jiacheng Li, Zhankui He, An Yan, Xiusi Chen, and Julian McAuley. *Bridging
  Language and Items for Retrieval and Recommendation.* arXiv preprint, 2024.
  [arXiv:2403.03952](https://arxiv.org/abs/2403.03952)